# IPL Intelligence — 04. Team Strategy, Venue & Match Context

Explore team scoring phases, venues, innings patterns and match-level outcomes where available. Verify field availability before drawing conclusions; some historical seasons and metadata may be incomplete.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "deliveries.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
df = pd.read_csv(DATA / "deliveries.csv", low_memory=False)
matches = pd.read_csv(DATA / "matches.csv", low_memory=False)
for c in ["start_date"]:
    if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")
for c in ["runs_off_bat","extras","total_runs","legal_ball","is_wicket","is_bowler_credited_wicket","over_number"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["season"] = df["season"].astype(str)
print(f"Deliveries: {len(df):,} | Matches: {df.match_id.nunique():,} | Seasons: {df.season.nunique()}")

## 1. Team performance by phase

In [ ]:
team_phase = df.groupby(["batting_team","phase"],as_index=False).agg(runs=("total_runs","sum"), legal_balls=("legal_ball","sum"), wickets=("is_wicket","sum"), matches=("match_id","nunique"))
team_phase["run_rate"] = np.where(team_phase.legal_balls>0,6*team_phase.runs/team_phase.legal_balls,np.nan)
display(team_phase.sort_values(["batting_team","phase"]))
wide = team_phase.pivot(index="batting_team",columns="phase",values="run_rate")
display(wide.round(2))

## 2. Team phase heatmap (matplotlib)

In [ ]:
fig, ax = plt.subplots(figsize=(10,max(5,len(wide)*.35))); im=ax.imshow(wide.fillna(0).values,aspect="auto"); ax.set_yticks(range(len(wide.index))); ax.set_yticklabels(wide.index); ax.set_xticks(range(len(wide.columns))); ax.set_xticklabels(wide.columns,rotation=20,ha="right"); ax.set_title("Team run rate by phase"); fig.colorbar(im,ax=ax,label="Runs per over"); fig.tight_layout()

## 3. Venue scoring environment

In [ ]:
venue = df.groupby("venue",dropna=False,as_index=False).agg(runs=("total_runs","sum"), legal_balls=("legal_ball","sum"), matches=("match_id","nunique"), wickets=("is_wicket","sum"))
venue["run_rate"] = np.where(venue.legal_balls>0,6*venue.runs/venue.legal_balls,np.nan)
venue = venue[venue.matches>=20].sort_values("run_rate",ascending=False)
display(venue.head(20).round(2))
fig, ax=plt.subplots(figsize=(10,6)); top=venue.head(15).sort_values("run_rate"); ax.barh(top.venue,top.run_rate); ax.set_xlabel("Runs per over"); ax.set_title("Highest scoring venues (minimum 20 matches)"); fig.tight_layout()

## 4. Match metadata audit

In [ ]:
display(matches.head())
print("Match metadata rows:",len(matches))
print("Metadata columns:",list(matches.columns))
for c in ["outcome","winner","byes","method","player_of_match","venue","date","season"]:
    if c in matches:
        print(f"{c}: missing={matches[c].isna().sum():,}; unique={matches[c].nunique(dropna=True):,}")

## 5. Optional match outcome analysis

In [ ]:
# Only run this if winner/outcome fields exist in the metadata archive.
if "winner" in matches.columns:
    winners=matches["winner"].dropna().value_counts().rename_axis("team").reset_index(name="wins")
    display(winners.head(20))
else:
    print("No winner field in match metadata; do not infer winners from innings totals without handling ties, DLS, super overs, and incomplete matches.")

## Research questions to take forward
- Which teams score fastest in the powerplay, and which accelerate most at the death?
- Which venues have higher scoring rates after controlling for match count and season?
- Are phase comparisons stable across eras?
- For stronger causal claims, use match-level context and control for season, venue, toss, opposition, innings, and target.